In [ ]:
# this cell is tagged parameters

PYLIB_DIR = None

# Reference info
REF_gtf_file = None
REF_quant_file = None

# Tool-agnostic registry-driven inputs.
# Populated by bmark_nb_runner.py from tool_registry.yaml.
# Schema: { name: {"quant": <path>, "gtf": <path-or-None>,
#                  "color": <str>, "family": <str>, "venn": <bool>} }
program_files = {}

# Whether downstream Venn-mode logic should restrict the seed set
# to entries with venn:true (set by --Venn_mode on the runner).
IGNORE_NONUNIQUE_NONREF = False


In [ ]:
import sys, os, re
sys.path.insert(0, PYLIB_DIR)


In [ ]:
import BenchmarkingRoutines
from importlib import reload
reload(BenchmarkingRoutines)
from BenchmarkingRoutines import *

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# colors for plots -- driven by registry entries
for _name, _info in program_files.items():
    set_color_palette(_name, _info["color"], "solid")


In [ ]:
# Build the quant-only truth universe from the full reference annotation.
# Truth quants are left-joined so reference-supported splice patterns
# absent from the truth quant table are treated as zero-expression.
# Restrict scoring to multi-exon reference isoforms whose genes have
# nonzero truth expression, while retaining zero-expression isoforms
# within those expressed genes.
ref_intron_df = parseGTFtoIntronIDs(REF_gtf_file)
ref_count_df = parseQuantsTSV(REF_quant_file)

import re

attr_re = re.compile(r'(\S+) "([^"]+)"')
tx_gene_rows = []
seen_transcript_ids = set()
with open(REF_gtf_file) as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        vals = line.rstrip("\n").split("\t")
        if len(vals) < 9 or vals[2] != "exon":
            continue
        attrs = dict(attr_re.findall(vals[8]))
        transcript_id = attrs.get("transcript_id")
        gene_id = attrs.get("gene_id")
        if transcript_id is None or gene_id is None or transcript_id in seen_transcript_ids:
            continue
        seen_transcript_ids.add(transcript_id)
        tx_gene_rows.append((transcript_id, gene_id))

ref_tx_gene_df = pd.DataFrame(tx_gene_rows, columns=["transcript_id", "gene_id"])
expressed_gene_ids = set(
    ref_tx_gene_df.merge(ref_count_df, how="left", on="transcript_id")
    .fillna({"tpm": 0})
    .groupby("gene_id", dropna=False)["tpm"]
    .sum()
    .loc[lambda s: s > 0]
    .index
)

ref_truth_df = ref_intron_df.merge(ref_count_df, how="left", on="transcript_id").fillna({"tpm": 0})
ref_truth_df = ref_truth_df[ref_truth_df["gene_id"].isin(expressed_gene_ids)].copy()
i_ref_df = indexDfByIntronId(ref_truth_df)
i_ref_df

In [ ]:
 i_ref_df["tpm"] = i_ref_df["tpm"] / i_ref_df["tpm"].sum() * 1e6

In [ ]:
i_ref_df.copy().reset_index().to_csv("refDf.intron_ids_and_expression.tsv", sep="\t", index=False)

In [ ]:
quant_only_dir = "processed_prog_results"

# Derive prog_quant_files from the registry-built program_files dict.
# Entries declaring gtf_source: own carry their own GTF; the rest fall
# back to REF_gtf for intron derivation.
prog_quant_files = {
    name: [info["quant"], info["gtf"] if info.get("gtf") else REF_gtf_file]
    for name, info in program_files.items()
}

# Cache per-GTF intron tables. Most entries reuse REF_gtf, which is
# expensive to parse for large genomes (GRCh38 takes minutes); without
# caching the per-program loop re-parses it once per entry.
_intronDf_cache = {}
def _get_intronDf(gtf_fname):
    if gtf_fname not in _intronDf_cache:
        _intronDf_cache[gtf_fname] = parseGTFtoIntronIDs(gtf_fname)
    return _intronDf_cache[gtf_fname]

fullQuantsDf_dict = {}
ref_intron_ids = i_ref_df.index
for progname, (tsv_fname, gtf_fname) in prog_quant_files.items():
    if tsv_fname is None:
        continue

    print(progname, tsv_fname, gtf_fname)
    intronDf = _get_intronDf(gtf_fname)
    countDf = parseQuantsTSV(tsv_fname)
    merged = intronDf.merge(countDf, how="inner", on="transcript_id")
    prog_i_df = indexDfByIntronId(merged)
    prog_i_df = prog_i_df.reindex(ref_intron_ids)
    prog_i_df["tpm"] = prog_i_df["tpm"].fillna(0)
    for meta_col in ("transcript_ids", "gene_ids"):
        if meta_col in prog_i_df.columns:
            prog_i_df[meta_col] = prog_i_df[meta_col].fillna("")
    fullQuantsDf_dict[progname] = prog_i_df

renormalized_fullQuantsDf_dict = {
    progname: renormalize_tpm_columns(df, ["tpm"])
    for progname, df in fullQuantsDf_dict.items()
}

progname_to_i_sample_df_dict_to_tsv(fullQuantsDf_dict, "progname_to_IntronId_expr_vals.tsv")
progname_to_i_sample_df_dict_to_tsv(renormalized_fullQuantsDf_dict, "progname_to_IntronId_expr_vals.renormalized.tsv")


In [ ]:
scatterplot_adj(i_ref_df, fullQuantsDf_dict)

In [ ]:
ma_plot_adj(i_ref_df, fullQuantsDf_dict)

In [ ]:
spearman_df = cor_spearman_barplot(i_ref_df, fullQuantsDf_dict)
spearman_df.to_csv("spearman_expr_cor.tsv", sep="\t", quoting=csv.QUOTE_NONE)

In [ ]:
pearson_df = cor_pearson_barplot(i_ref_df, fullQuantsDf_dict)
pearson_df.to_csv("pearson_expr_cor.tsv", sep="\t", quoting=csv.QUOTE_NONE)

In [ ]:
median_rel_diff_df = rel_diff_barplot(i_ref_df, fullQuantsDf_dict, 'median')
median_rel_diff_df.to_csv("median_rel_diff.tsv", sep="\t", quoting=csv.QUOTE_NONE)

In [ ]:
mean_rel_diff_df = rel_diff_barplot(i_ref_df, fullQuantsDf_dict, 'mean')
mean_rel_diff_df.to_csv("mean_rel_diff.tsv", sep="\t", quoting=csv.QUOTE_NONE)

In [ ]:
oarfish_style_metrics_df = oarfish_style_metrics_table(i_ref_df, fullQuantsDf_dict)
oarfish_style_metrics_df.to_csv("oarfish_style_metrics.tsv", sep="\t", index=False, quoting=csv.QUOTE_NONE)

In [ ]:
rel_diff_vs_expr_percentile_plot(i_ref_df, fullQuantsDf_dict, 33, 'median',
                                 'all ref-reduced sets, all ref transcripts')

In [ ]:
def _primary_gene_id(gene_ids):
    if pd.isna(gene_ids):
        return ""
    ids = sorted({g.strip() for g in str(gene_ids).split(",") if g.strip()})
    return ",".join(ids)


def _safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=float),
        where=denominator > 0,
    )


def isoform_gene_fraction_agreement(
    i_ref_df,
    progname_to_df_dict,
    min_ref_isoforms_per_gene=1,
    gene_set_label="all_ref_genes",
):
    ref_fraction_df = i_ref_df[["tpm", "transcript_ids", "gene_ids"]].copy()
    ref_fraction_df["ref_gene_id"] = ref_fraction_df["gene_ids"].apply(_primary_gene_id)
    ref_fraction_df = ref_fraction_df[ref_fraction_df["ref_gene_id"] != ""].copy()
    ref_fraction_df["ref_gene_isoform_count"] = ref_fraction_df.groupby("ref_gene_id")[
        "ref_gene_id"
    ].transform("size")
    ref_fraction_df = ref_fraction_df[
        ref_fraction_df["ref_gene_isoform_count"] >= min_ref_isoforms_per_gene
    ].copy()
    ref_fraction_df["ref_gene_tpm"] = ref_fraction_df.groupby("ref_gene_id")[
        "tpm"
    ].transform("sum")
    ref_fraction_df["ref_isoform_gene_fraction"] = _safe_divide(
        ref_fraction_df["tpm"], ref_fraction_df["ref_gene_tpm"]
    )
    ref_fraction_df = ref_fraction_df.rename(columns={"tpm": "ref_tpm"})

    detail_dfs = []
    summary_rows = []
    for progname, prog_df in progname_to_df_dict.items():
        detail_df = ref_fraction_df.copy()
        detail_df["progname"] = progname
        detail_df["tpm"] = prog_df["tpm"].reindex(detail_df.index).fillna(0).astype(float)
        detail_df["gene_tpm"] = detail_df.groupby("ref_gene_id")["tpm"].transform("sum")
        detail_df["isoform_gene_fraction"] = _safe_divide(
            detail_df["tpm"], detail_df["gene_tpm"]
        )
        detail_df["fraction_error"] = (
            detail_df["isoform_gene_fraction"]
            - detail_df["ref_isoform_gene_fraction"]
        )
        detail_df["abs_fraction_error"] = detail_df["fraction_error"].abs()
        detail_dfs.append(detail_df)

        gene_tvd_df = (
            detail_df.groupby("ref_gene_id")
            .agg(
                gene_fraction_tvd=("abs_fraction_error", lambda s: 0.5 * s.sum()),
                ref_gene_tpm=("ref_gene_tpm", "first"),
                gene_tpm=("gene_tpm", "first"),
            )
            .reset_index()
        )
        gene_tvd_df.loc[gene_tvd_df["gene_tpm"] <= 0, "gene_fraction_tvd"] = 1.0

        weight_sum = gene_tvd_df["ref_gene_tpm"].sum()
        weighted_gene_tvd = np.nan
        if weight_sum > 0:
            weighted_gene_tvd = np.average(
                gene_tvd_df["gene_fraction_tvd"], weights=gene_tvd_df["ref_gene_tpm"]
            )

        summary_rows.append(
            {
                "progname": progname,
                "gene_set": gene_set_label,
                "family": program_files.get(progname, {}).get("family", ""),
                "color": program_files.get(progname, {}).get("color", ""),
                "venn": program_files.get(progname, {}).get("venn", ""),
                "n_isoforms": len(detail_df),
                "n_genes": gene_tvd_df.shape[0],
                "gene_detected_fraction": (gene_tvd_df["gene_tpm"] > 0).mean(),
                "isoform_fraction_mae": detail_df["abs_fraction_error"].mean(),
                "isoform_fraction_median_abs_error": detail_df[
                    "abs_fraction_error"
                ].median(),
                "isoform_fraction_rmse": np.sqrt(
                    np.mean(np.square(detail_df["fraction_error"]))
                ),
                "isoform_fraction_pearson": safe_pearson(
                    detail_df["ref_isoform_gene_fraction"],
                    detail_df["isoform_gene_fraction"],
                ),
                "isoform_fraction_spearman": safe_spearman(
                    detail_df["ref_isoform_gene_fraction"],
                    detail_df["isoform_gene_fraction"],
                ),
                "gene_fraction_tvd_mean": gene_tvd_df["gene_fraction_tvd"].mean(),
                "gene_fraction_tvd_median": gene_tvd_df[
                    "gene_fraction_tvd"
                ].median(),
                "gene_fraction_tvd_expr_weighted_mean": weighted_gene_tvd,
            }
        )

    detail_df = pd.concat(detail_dfs).reset_index()
    summary_df = pd.DataFrame(summary_rows).sort_values(
        by=["gene_fraction_tvd_expr_weighted_mean", "isoform_fraction_mae"]
    )
    return detail_df, summary_df


isoform_gene_fraction_values_df, isoform_gene_fraction_summary_all_df = isoform_gene_fraction_agreement(
    i_ref_df, fullQuantsDf_dict, min_ref_isoforms_per_gene=1, gene_set_label="all_ref_genes"
)
_, isoform_gene_fraction_summary_multi_df = isoform_gene_fraction_agreement(
    i_ref_df, fullQuantsDf_dict, min_ref_isoforms_per_gene=2, gene_set_label="multi_isoform_ref_genes"
)

isoform_gene_fraction_summary_df = pd.concat(
    [isoform_gene_fraction_summary_all_df, isoform_gene_fraction_summary_multi_df]
).sort_values(by=["gene_set", "gene_fraction_tvd_expr_weighted_mean", "isoform_fraction_mae"])

isoform_gene_fraction_values_df.to_csv(
    "isoform_gene_fraction_values.tsv", sep="\t", index=False, quoting=csv.QUOTE_NONE
)
isoform_gene_fraction_summary_df.to_csv(
    "isoform_gene_fraction_summary.tsv", sep="\t", index=False, quoting=csv.QUOTE_NONE
)
isoform_gene_fraction_summary_df


In [ ]:
def _gene_set_title(gene_set):
    return gene_set.replace("_", " ").replace("ref", "reference")


def plot_isoform_gene_fraction_rankings(summary_df, gene_set, output_prefix="isoform_gene_fraction"):
    plot_df = summary_df[summary_df["gene_set"] == gene_set].copy()
    plot_df = plot_df.sort_values(
        by=["gene_fraction_tvd_expr_weighted_mean", "isoform_fraction_mae"]
    ).reset_index(drop=True)
    plot_df["rank"] = np.arange(1, len(plot_df) + 1)

    height = max(4.5, 0.28 * len(plot_df) + 1.2)
    y_pos = np.arange(len(plot_df))
    y_labels = plot_df["rank"].astype(str) + ". " + plot_df["progname"]

    fig, ax = plt.subplots(figsize=(7.2, height), dpi=150, layout="constrained")
    bars = ax.barh(
        y_pos,
        plot_df["gene_fraction_tvd_expr_weighted_mean"],
        color=plot_df["color"],
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels)
    ax.invert_yaxis()
    ax.set_xlabel("Expression-weighted per-gene isoform-fraction TVD")
    ax.set_title(_gene_set_title(gene_set))
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.bar_label(bars, labels=[f"{v:.4f}" for v in plot_df["gene_fraction_tvd_expr_weighted_mean"]], fontsize=7, padding=2)

    rank_png = f"{output_prefix}_ranking.{gene_set}.png"
    rank_pdf = f"{output_prefix}_ranking.{gene_set}.pdf"
    fig.savefig(rank_png, bbox_inches="tight")
    fig.savefig(rank_pdf, bbox_inches="tight")

    metric_specs = [
        ("gene_fraction_tvd_expr_weighted_mean", "Expr-weighted\ngene TVD", "lower"),
        ("gene_fraction_tvd_mean", "Mean\ngene TVD", "lower"),
        ("isoform_fraction_mae", "Isoform\nfraction MAE", "lower"),
        ("isoform_fraction_spearman", "Isoform\nfraction Spearman", "higher"),
    ]
    fig2, axs = plt.subplots(
        ncols=len(metric_specs),
        nrows=1,
        figsize=(10.5, height),
        dpi=150,
        layout="constrained",
        sharey=True,
    )
    for ax_i, (metric, label, direction) in zip(axs, metric_specs):
        bars = ax_i.barh(y_pos, plot_df[metric], color=plot_df["color"])
        ax_i.set_yticks(y_pos)
        ax_i.set_yticklabels(y_labels)
        ax_i.set_xlabel(label)
        ax_i.set_title(direction + " is better", fontsize=8)
        ax_i.spines["right"].set_visible(False)
        ax_i.spines["top"].set_visible(False)
        value_labels = [f"{v:.4f}" for v in plot_df[metric]]
        ax_i.bar_label(bars, labels=value_labels, fontsize=6, padding=2)
        if metric == "isoform_fraction_spearman":
            xmin = max(0, plot_df[metric].min() - 0.02)
            ax_i.set_xlim(xmin, 1.0)

    axs[0].invert_yaxis()
    fig2.suptitle(_gene_set_title(gene_set), fontsize=11)
    panel_png = f"{output_prefix}_metric_panel.{gene_set}.png"
    panel_pdf = f"{output_prefix}_metric_panel.{gene_set}.pdf"
    fig2.savefig(panel_png, bbox_inches="tight")
    fig2.savefig(panel_pdf, bbox_inches="tight")

    return plot_df, [rank_png, rank_pdf, panel_png, panel_pdf]


isoform_gene_fraction_ranked_dfs = {}
isoform_gene_fraction_plot_files = []
for gene_set in ["all_ref_genes", "multi_isoform_ref_genes"]:
    ranked_df, plot_files = plot_isoform_gene_fraction_rankings(
        isoform_gene_fraction_summary_df, gene_set
    )
    isoform_gene_fraction_ranked_dfs[gene_set] = ranked_df
    isoform_gene_fraction_plot_files.extend(plot_files)

isoform_gene_fraction_plot_files
